# gold_kpis

Camada **Gold** de indicadores mensais da distribuidora: consolida em uma única série temporal (48 meses) as métricas da diretoria para o painel executivo e a comparação entre períodos (**P1** × **P2**).

- **Entrada:** `gold.dim_mes`, `gold.fato_alimentador_mes` e `gold.fato_uc_mes` (`gold_base`), `silver.tb_ocorrencias_rede` e `silver.tb_atendimentos` (`silver_rede_atendimento`) e `silver.tb_inspecoes` (`silver_perdas`).
- **Saída:** `gold.agg_kpis_mensal`.
- **Alimenta:** `GET /dashboard` (indicadores do mês e comparação P1 × P2), agente de IA e diagnóstico executivo.
- Segue o documento *Decisões da Engenharia de Dados: camada Gold* (v2, seções 2 a 5). Idempotente: grava em `overwrite` e pode ser reexecutado.

| Tabela | Grão (1 linha por) | Conteúdo |
|---|---|---|
| `agg_kpis_mensal` | mês (`mes_referencia`, 48 meses) | calendário/período (`P1`/`P2`), bandeira, UCs atendidas no mês, receita, consumo válido, tarifa média, inadimplência, DEC/FEC global (total e sem dia crítico), atendimento/satisfação e inspeções/energia recuperada |

**Métricas e regras de negócio (seções 4 e 5)**
- **Faturamento e medição (`gold.fato_uc_mes`):**
  - `receita_total` = Σ `valor_total`; `receita_energia` = Σ `valor_energia`; `tributos` = Σ `tributos`; `consumo_kwh` = Σ `consumo_kwh_valido` (todos em `decimal(14,2)`).
  - `tarifa_media` = `receita_energia` ÷ `consumo_kwh` em `decimal(14,6)` (`try_divide`; nula se `consumo_kwh` = 0).
  - `inadimplencia` = `valor_em_aberto` (`status == 'EM ABERTO'`) ÷ `receita_total` em `decimal(14,4)` (`try_divide`).
  - **Uso explícito das flags (regra 5):** o volume `consumo_kwh` soma `consumo_kwh_valido` da `fato_uc_mes` (consumo sem os 30 registros negativos intratáveis marcados com `flag_consumo_negativo`, já com a recuperação da 1ª leitura pós-troca e do sinal invertido feita na `silver_medicao`), contabilizando à parte `leituras_estimadas` (`flag_leitura_estimada`).
- **Continuidade (`silver.tb_ocorrencias_rede` + `gold.fato_alimentador_mes`):**
  - Mês definido pela `data_abertura` (`yyyy-MM`), com as mesmas regras do `gold_dec_fec`: interrupções longas (`duracao_min >= 3`, PRODIST Módulo 8), excluindo `flag_sem_interrupcao` (`OSCILACAO` não interrompe o fornecimento) e `flag_antes_ligacao` (antes da ligação a UC não existia).
  - Tempo sem energia medido por `duracao_efetiva_min` (sem contar em dobro o tempo sobreposto); interrupções inteiramente contidas dentro de outra da mesma UC (`duracao_efetiva_min == 0`, 147 casos) também não contam duas vezes no FEC (`duracao_efetiva_min > 0`). Denominador $C_c$ por mês (`ucs_atendidas` = UCs ligadas até o fim do mês, somadas nos alimentadores da `gold.fato_alimentador_mes`).
  - No agregado global do mês: `dec` = $\Sigma (\text{duracao\_efetiva\_min}) \div (60 \times C_c)$ e `fec` = $\text{interrupcoes} \div C_c$ em `decimal(14,4)`, além das versões expurgando dias críticos (`dec_sem_dia_critico` e `fec_sem_dia_critico`, onde `~flag_dia_critico`). As colunas `interrupcoes` e `horas_interrupcao` têm os mesmos nomes da `gold.agg_dec_fec_mensal`.
  - **Uso explícito das flags (regra 5):** contagens de `ocorrencias_sobrepostas` (`flag_sobreposicao`), `ocorrencias_teto_48h` (`flag_duracao_teto`) e `ocorrencias_dia_critico` (`flag_dia_critico`) expostas no mês para transparência analítica.
- **Atendimento (`silver.tb_atendimentos`) e Inspeções (`silver.tb_inspecoes`):**
  - `satisfacao_media` = média de `nota_satisfacao` em `decimal(14,2)`, ignorando nulos (nunca convertida para zero).
  - `taxa_acerto_inspecao` = `inspecoes_confirmadas` (`resultado == 'IRREGULARIDADE CONFIRMADA'`) ÷ `inspecoes` em `decimal(14,4)` (`try_divide`; fica nula nos meses sem inspeção). `kwh_recuperado` = Σ `kwh_recuperado` em `decimal(14,2)` (0 nos meses sem inspeção).

In [0]:
%run ../bronze-to-silver/funcoes_comuns

In [0]:
dbutils.widgets.text("catalog", "workspace")
CATALOGO = dbutils.widgets.get("catalog")

In [0]:
# ---------- Regras de negócio: KPIs mensais ----------
STATUS_ABERTO = "EM ABERTO"
RESULTADO_CONFIRMADO = "IRREGULARIDADE CONFIRMADA"
DURACAO_MINIMA_DEC_MIN = 3       # PRODIST: interrupção longa = 3 minutos ou mais
# Fora do DEC/FEC (a linha fica na Silver): OSCILACAO não interrompe o fornecimento; antes da ligação, a UC não existia
FLAGS_EXCLUI_DEC = ["flag_sem_interrupcao", "flag_antes_ligacao"]

In [0]:
# Leitura da gold_base (dimensões e fato central) e das Silver de operação/atendimento/inspeções
dim_mes = spark.table(f"{CATALOGO}.gold.dim_mes").drop("_atualizado_em")
fato = spark.table(f"{CATALOGO}.gold.fato_uc_mes").drop("_atualizado_em")

oc = spark.table(f"{CATALOGO}.silver.tb_ocorrencias_rede")
at = spark.table(f"{CATALOGO}.silver.tb_atendimentos")
ins = spark.table(f"{CATALOGO}.silver.tb_inspecoes")

# Cc do DEC/FEC global de cada mês: UCs ligadas até o fim do mês, somadas nos alimentadores (gold_base)
cc_mes = (spark.table(f"{CATALOGO}.gold.fato_alimentador_mes").groupBy("mes_referencia")
            .agg(F.sum("ucs_atendidas").alias("ucs_atendidas")))   # o gold_base garante que nunca é zero

In [0]:
# 1. Faturamento, consumo e inadimplência por mês (a partir de gold.fato_uc_mes)
dec2 = lambda c: F.col(c).cast("decimal(14,2)")
em_aberto = F.col("status") == STATUS_ABERTO
consumo_valido = F.col("consumo_kwh_valido")   # gold_base: consumo sem os negativos marcados (flag_consumo_negativo)

fat_mes = (fato.groupBy("mes_referencia")
    .agg(
        F.count("*").alias("faturas"),
        F.sum("valor_total").alias("receita_total"),
        F.sum("valor_energia").alias("receita_energia"),
        F.sum("tributos").alias("tributos"),
        F.coalesce(F.sum(consumo_valido), F.lit(0)).alias("consumo_kwh"),
        F.sum(em_aberto.cast("long")).alias("faturas_em_aberto"),
        F.coalesce(F.sum(F.when(em_aberto, F.col("valor_total"))), F.lit(0)).alias("valor_em_aberto"),
        F.sum(F.col("flag_leitura_estimada").cast("long")).alias("leituras_estimadas"),
    )
    .withColumns({c: dec2(c) for c in ["receita_total", "receita_energia", "tributos", "consumo_kwh", "valor_em_aberto"]}))

In [0]:
# 2. Continuidade (DEC e FEC) por mês de abertura: as mesmas regras do gold_dec_fec
excluida = F.array_contains(F.array(*FLAGS_EXCLUI_DEC), True)
fora_dia_critico = ~F.col("flag_dia_critico")
# Interrupção que fica inteira dentro de outra da mesma UC (duração efetiva 0) é o mesmo apagão
# registrado duas vezes: não conta no DEC (já não somava minutos) nem no FEC (147 casos)
conta_no_fec = F.col("duracao_efetiva_min") > 0
oc_validas = (oc.filter((F.col("duracao_min") >= DURACAO_MINIMA_DEC_MIN) & ~excluida)
                .withColumn("mes_referencia", F.date_format("data_abertura", "yyyy-MM")))

oc_mes = (oc_validas.groupBy("mes_referencia")
    .agg(
        F.sum(conta_no_fec.cast("long")).alias("interrupcoes"),
        F.sum("duracao_efetiva_min").alias("__minutos"),   # sem o tempo sobreposto contado em dobro
        F.sum(F.when(fora_dia_critico, F.col("duracao_efetiva_min")).otherwise(0)).alias("__minutos_sem_dc"),
        F.sum((conta_no_fec & fora_dia_critico).cast("long")).alias("__interrupcoes_sem_dc"),
        F.sum(F.col("flag_sobreposicao").cast("long")).alias("ocorrencias_sobrepostas"),
        F.sum(F.col("flag_duracao_teto").cast("long")).alias("ocorrencias_teto_48h"),
        F.sum(F.col("flag_dia_critico").cast("long")).alias("ocorrencias_dia_critico"),
    ))

In [0]:
# 3. Atendimento e satisfação média por mês de abertura (média ignora nulos; nunca vira zero)
at_mes = (at.withColumn("mes_referencia", F.date_format("data_abertura", "yyyy-MM"))
    .groupBy("mes_referencia")
    .agg(
        F.count("*").alias("atendimentos"),
        F.avg("nota_satisfacao").cast("decimal(14,2)").alias("satisfacao_media"),
    ))

# 4. Inspeções, taxa de acerto e energia recuperada por mês da inspeção
confirmada = F.col("resultado") == RESULTADO_CONFIRMADO
ins_mes = (ins.withColumn("mes_referencia", F.date_format("data", "yyyy-MM"))
    .groupBy("mes_referencia")
    .agg(
        F.count("*").alias("inspecoes"),
        F.sum(confirmada.cast("long")).alias("inspecoes_confirmadas"),
        F.coalesce(F.sum("kwh_recuperado"), F.lit(0)).cast("decimal(14,2)").alias("kwh_recuperado"),
    ))

In [0]:
# 5. Montagem de gold.agg_kpis_mensal (1 linha por mês a partir de dim_mes)
# Somas e contagens de meses sem evento viram 0; razões/médias sem base ficam nulas
COLS_ZERO = [
    "faturas", "receita_total", "receita_energia", "tributos", "consumo_kwh",
    "faturas_em_aberto", "valor_em_aberto", "leituras_estimadas",
    "interrupcoes", "ocorrencias_sobrepostas", "ocorrencias_teto_48h", "ocorrencias_dia_critico",
    "__minutos", "__minutos_sem_dc", "__interrupcoes_sem_dc",
    "atendimentos", "inspecoes", "inspecoes_confirmadas", "kwh_recuperado",
]

por_cc = lambda c, fator=1: F.try_divide(F.col(c), F.col("ucs_atendidas") * fator)

kpis = (dim_mes
    .join(cc_mes, "mes_referencia")
    .join(fat_mes, "mes_referencia", "left")
    .join(oc_mes, "mes_referencia", "left")
    .join(at_mes, "mes_referencia", "left")
    .join(ins_mes, "mes_referencia", "left")
    .fillna(0, subset=COLS_ZERO)
    .withColumns({
        "horas_interrupcao": (F.col("__minutos") / 60).cast("decimal(14,2)"),
        "dec": por_cc("__minutos", 60).cast("decimal(14,4)"),
        "fec": por_cc("interrupcoes").cast("decimal(14,4)"),
        "dec_sem_dia_critico": por_cc("__minutos_sem_dc", 60).cast("decimal(14,4)"),
        "fec_sem_dia_critico": por_cc("__interrupcoes_sem_dc").cast("decimal(14,4)"),
        "tarifa_media": F.try_divide("receita_energia", "consumo_kwh").cast("decimal(14,6)"),
        "inadimplencia": F.try_divide("valor_em_aberto", "receita_total").cast("decimal(14,4)"),
        "taxa_acerto_inspecao": F.try_divide("inspecoes_confirmadas", "inspecoes").cast("decimal(14,4)"),
    })
    .select(
        "mes_referencia", "ano", "mes", "trimestre", "periodo", "bandeira_tarifaria", "ucs_atendidas",
        "faturas", "receita_total", "receita_energia", "tributos", "consumo_kwh", "tarifa_media",
        "faturas_em_aberto", "valor_em_aberto", "inadimplencia", "leituras_estimadas",
        "interrupcoes", "horas_interrupcao", "dec", "fec", "dec_sem_dia_critico", "fec_sem_dia_critico",
        "ocorrencias_sobrepostas", "ocorrencias_teto_48h", "ocorrencias_dia_critico",
        "atendimentos", "satisfacao_media",
        "inspecoes", "inspecoes_confirmadas", "taxa_acerto_inspecao", "kwh_recuperado",
    ))

In [0]:
# 6. Checagens e reconciliação: se algo falhar, o notebook para ANTES de gravar
checar_unicidade(kpis, ["mes_referencia"], "agg_kpis_mensal")
checar_sem_nulos(kpis, ["mes_referencia", "ano", "mes", "trimestre", "bandeira_tarifaria", "ucs_atendidas",
                        *[c for c in COLS_ZERO if not c.startswith("__")],
                        "horas_interrupcao", "dec", "fec", "dec_sem_dia_critico", "fec_sem_dia_critico"],
                 "agg_kpis_mensal")
assert kpis.count() == dim_mes.count(), f"esperado {dim_mes.count()} meses, veio {kpis.count()}"

# Confere que nenhuma ocorrência ou atendimento ficou fora dos meses da dim_mes
oc_fora = oc_validas.join(dim_mes, "mes_referencia", "left_anti").count()
at_fora = (at.withColumn("mes_referencia", F.date_format("data_abertura", "yyyy-MM"))
             .join(dim_mes, "mes_referencia", "left_anti").count())
assert oc_fora == 0 and at_fora == 0, f"eventos fora de dim_mes: ocorrencias={oc_fora}, atendimentos={at_fora}"

# Reconciliação com gold.fato_uc_mes e com as tabelas Silver
g = kpis.agg(
    F.sum("receita_total").alias("rec"), F.sum("receita_energia").alias("rec_en"),
    F.sum("tributos").alias("trib"), F.sum("consumo_kwh").alias("kwh"),
    F.sum("faturas").alias("fat"), F.sum("valor_em_aberto").alias("aberto"),
    F.sum("interrupcoes").alias("oc"), F.sum("atendimentos").alias("at"),
    F.sum("inspecoes").alias("ins"), F.sum("kwh_recuperado").alias("kwh_rec"),
).first()

f = fato.agg(
    F.sum("valor_total").alias("rec"), F.sum("valor_energia").alias("rec_en"),
    F.sum("tributos").alias("trib"),
    F.coalesce(F.sum(consumo_valido), F.lit(0)).cast("decimal(14,2)").alias("kwh"),
    F.count("*").alias("fat"),
    F.coalesce(F.sum(F.when(em_aberto, F.col("valor_total"))), F.lit(0)).cast("decimal(14,2)").alias("aberto"),
).first()

ins_no_periodo = (ins.withColumn("mes_referencia", F.date_format("data", "yyyy-MM"))
                     .join(dim_mes.select("mes_referencia"), "mes_referencia"))
ins_ref = ins_no_periodo.agg(
    F.count("*").alias("ins"),
    F.coalesce(F.sum("kwh_recuperado"), F.lit(0)).cast("decimal(14,2)").alias("kwh_rec"),
).first()

assert (g["rec"], g["rec_en"], g["trib"], g["kwh"], g["fat"], g["aberto"]) == (
    f["rec"], f["rec_en"], f["trib"], f["kwh"], f["fat"], f["aberto"]
), f"reconciliação com fato_uc_mes não fecha: gold={g} fato={f}"
n_oc_fec = oc_validas.filter(conta_no_fec).count()
assert g["oc"] == n_oc_fec, f"interrupções não fecham: gold={g['oc']} silver={n_oc_fec}"
assert g["at"] == at.count(), f"atendimentos não fecham: gold={g['at']} silver={at.count()}"
assert (g["ins"], g["kwh_rec"]) == (ins_ref["ins"], ins_ref["kwh_rec"]), (
    f"inspeções não fecham: gold={(g['ins'], g['kwh_rec'])} silver={(ins_ref['ins'], ins_ref['kwh_rec'])}"
)

print(f"Reconciliação OK | receita R$ {g['rec']:,.2f} | kWh {g['kwh']:,.2f} | "
      f"interrupções {g['oc']:,} | atendimentos {g['at']:,} | inspeções {g['ins']:,}")

In [0]:
# 7. Gravar (overwrite: reexecutável) e exibir resumo comparativo P1 × P2
(kpis.withColumn("_atualizado_em", F.current_timestamp())
     .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
     .saveAsTable(f"{CATALOGO}.gold.agg_kpis_mensal"))
print(f"gold.agg_kpis_mensal: {spark.table(f'{CATALOGO}.gold.agg_kpis_mensal').count():,} linhas")

# Satisfação do período: média das notas do período inteiro. A média das médias mensais daria o mesmo peso
# a um mês com poucas notas e a um mês com muitas.
satisfacao_periodo = (at.withColumn("mes_referencia", F.date_format("data_abertura", "yyyy-MM"))
    .join(dim_mes.select("mes_referencia", "periodo"), "mes_referencia")
    .groupBy("periodo").agg(F.avg("nota_satisfacao").cast("decimal(14,2)").alias("satisfacao_media")))

resumo_periodos = (spark.table(f"{CATALOGO}.gold.agg_kpis_mensal")
    .filter(F.col("periodo").isNotNull())
    .groupBy("periodo")
    .agg(
        F.sum("receita_total").cast("decimal(14,2)").alias("receita_total"),
        F.sum("receita_energia").cast("decimal(14,2)").alias("receita_energia"),
        F.sum("consumo_kwh").cast("decimal(14,2)").alias("consumo_kwh"),
        F.try_divide(F.sum("receita_energia"), F.sum("consumo_kwh")).cast("decimal(14,6)").alias("tarifa_media"),
        F.try_divide(F.sum("valor_em_aberto"), F.sum("receita_total")).cast("decimal(14,4)").alias("inadimplencia"),
        F.sum("dec").cast("decimal(14,4)").alias("dec_acumulado"),
        F.sum("fec").cast("decimal(14,4)").alias("fec_acumulado"),
        F.sum("dec_sem_dia_critico").cast("decimal(14,4)").alias("dec_acumulado_sem_dia_critico"),
        F.sum("fec_sem_dia_critico").cast("decimal(14,4)").alias("fec_acumulado_sem_dia_critico"),
        F.sum("atendimentos").alias("atendimentos"),
    )
    .join(satisfacao_periodo, "periodo")
    .orderBy("periodo"))
display(resumo_periodos)
display(spark.table(f"{CATALOGO}.gold.agg_kpis_mensal").orderBy("mes_referencia"))